In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Cell 2: Chargement des données
users = pd.read_csv('../data/users_data.csv')
courses = pd.read_csv('../data/courses_data.csv')
matches = pd.read_csv('../data/match_data.csv')

print(f"Utilisateurs: {len(users)}")
print(f"Cours: {len(courses)}")
print(f"Matchs: {len(matches)}")

# Cell 3: Analyse exploratoire
users.head()
users.describe()

# Distribution des niveaux
plt.figure(figsize=(10, 6))
sns.countplot(data=users, x='skill_level')
plt.title('Distribution des niveaux de compétence')
plt.show()

# Distribution par jeu
plt.figure(figsize=(10, 6))
sns.countplot(data=users, y='favorite_game')
plt.title('Jeux favoris')
plt.show()

# Cell 4: Préparation des données pour le modèle de prédiction
features = ['games_played', 'win_rate', 'kda', 'hours_played']
X = users[features]
y = users['skill_level']

# Encodage des labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42
)

# Cell 5: Entraînement du modèle
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Prédictions
y_pred = rf_model.predict(X_test)

# Évaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision: {accuracy:.2f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Cell 6: Matrice de confusion
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Matrice de confusion')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

# Cell 7: Entraînement du modèle de recommandation
# Préparation des features pour KNN
user_features = pd.get_dummies(users[['skill_level', 'favorite_game']])
user_features['hours_played'] = users['hours_played']
user_features['win_rate'] = users['win_rate']

scaler_knn = StandardScaler()
user_features_scaled = scaler_knn.fit_transform(user_features)

# KNN
knn_model = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_model.fit(user_features_scaled)

# Test de recommandation
test_user_idx = 0
distances, indices = knn_model.kneighbors(
    user_features_scaled[test_user_idx].reshape(1, -1)
)

print("Utilisateur testé:", users.iloc[test_user_idx]['username'])
print("Utilisateurs similaires:")
for idx in indices[0][1:]:
    print(f"- {users.iloc[idx]['username']} (distance: {distances[0][idx]:.3f})")

# Cell 8: Sauvegarde des modèles
import joblib

joblib.dump({
    'model': rf_model,
    'label_encoder': le,
    'scaler': scaler,
    'feature_names': features
}, '../models/skill_predictor.pkl')

joblib.dump({
    'model': knn_model,
    'scaler': scaler_knn,
    'user_features': user_features_scaled,
    'users_df': users
}, '../models/recommendation_model.pkl')

print("Modèles sauvegardés avec succès !")